In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    log_loss,
    brier_score_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# Project root
project_root = Path.cwd().parents[2]

# V3.1 feature directory
feature_dir = (
    project_root
    / "data"
    / "processed"
    / "engineered_features"
    / "group_3_other_skill_history"
    / "v3.1"
)

# Prediction output directory
prediction_dir = (
    project_root
    / "data"
    / "processed"
    / "model_predictions"
)

prediction_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Feature directory:", feature_dir)
print("Prediction directory:", prediction_dir)

Project root: c:\Users\rajur\OneDrive\Desktop\next-move-ai
Feature directory: c:\Users\rajur\OneDrive\Desktop\next-move-ai\data\processed\engineered_features\group_3_other_skill_history\v3.1
Prediction directory: c:\Users\rajur\OneDrive\Desktop\next-move-ai\data\processed\model_predictions


In [2]:
v31_fold_datasets = []

for fold_number in range(1, 6):

    train_path = feature_dir / f"fold_{fold_number}_train.csv"
    val_path = feature_dir / f"fold_{fold_number}_validation.csv"

    train_df = pd.read_csv(train_path)
    val_df = pd.read_csv(val_path)

    v31_fold_datasets.append((train_df, val_df))

    print(
        f"Fold {fold_number}: "
        f"Train = {train_df.shape}, "
        f"Validation = {val_df.shape}"
    )

print("\nAll V3.1 datasets loaded successfully!")

Fold 1: Train = (177324, 4), Validation = (42659, 4)
Fold 2: Train = (176273, 4), Validation = (43710, 4)
Fold 3: Train = (170186, 4), Validation = (49797, 4)
Fold 4: Train = (178311, 4), Validation = (41672, 4)
Fold 5: Train = (177838, 4), Validation = (42145, 4)

All V3.1 datasets loaded successfully!


In [3]:
feature_column = "other_skill_smoothed_accuracy"
target_column = "correct"

for fold_number, (train_df, val_df) in enumerate(
    v31_fold_datasets, start=1
):
    print(f"\n========== Fold {fold_number} ==========")

    print("Training missing values:")
    print(train_df.isna().sum())

    print("\nValidation missing values:")
    print(val_df.isna().sum())

    print("\nTraining feature range:")
    print(train_df[feature_column].min(),
          train_df[feature_column].max())

    print("Validation feature range:")
    print(val_df[feature_column].min(),
          val_df[feature_column].max())

    print("\nTraining target distribution:")
    print(train_df[target_column].value_counts(normalize=True))

    print("\nValidation target distribution:")
    print(val_df[target_column].value_counts(normalize=True))


========== Fold 1 ==========
Training missing values:
user_id                          0
order_id                         0
correct                          0
other_skill_smoothed_accuracy    0
dtype: int64

Validation missing values:
user_id                          0
order_id                         0
correct                          0
other_skill_smoothed_accuracy    0
dtype: int64

Training feature range:
0.0337005285742449 1.0
Validation feature range:
0.0615807097922183 0.9351188478359068

Training target distribution:
correct
1    0.660136
0    0.339864
Name: proportion, dtype: float64

Validation target distribution:
correct
1    0.652336
0    0.347664
Name: proportion, dtype: float64

========== Fold 2 ==========
Training missing values:
user_id                          0
order_id                         0
correct                          0
other_skill_smoothed_accuracy    0
dtype: int64

Validation missing values:
user_id                          0
order_id                  

In [4]:
feature_column = ["other_skill_smoothed_accuracy"]
target_column = "correct"

v31_models = []
v31_fold_results = []

for fold_number, (train_df, val_df) in enumerate(
    v31_fold_datasets, start=1
):
    print(f"\nTraining Fold {fold_number}...")

    X_train = train_df[feature_column]
    y_train = train_df[target_column]

    X_val = val_df[feature_column]
    y_val = val_df[target_column]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logistic_regression", LogisticRegression(
            C=1.0,
            max_iter=1000,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)

    y_pred_proba = model.predict_proba(X_val)[:, 1]

    fold_log_loss = log_loss(y_val, y_pred_proba)
    fold_brier = brier_score_loss(y_val, y_pred_proba)

    v31_models.append(model)

    v31_fold_results.append({
        "fold": fold_number,
        "log_loss": fold_log_loss,
        "brier_score": fold_brier
    })

    print(f"Log Loss: {fold_log_loss:.4f}")
    print(f"Brier Score: {fold_brier:.4f}")

print("\nAll five V3.1 models trained successfully!")


Training Fold 1...
Log Loss: 0.6169
Brier Score: 0.2137

Training Fold 2...
Log Loss: 0.6123
Brier Score: 0.2114

Training Fold 3...
Log Loss: 0.6298
Brier Score: 0.2195

Training Fold 4...
Log Loss: 0.6181
Brier Score: 0.2141

Training Fold 5...
Log Loss: 0.6104
Brier Score: 0.2106

All five V3.1 models trained successfully!


In [5]:
v31_predictions = []

for fold_number, (train_df, val_df) in enumerate(
    v31_fold_datasets, start=1
):
    model = v31_models[fold_number - 1]

    X_val = val_df[["other_skill_smoothed_accuracy"]]

    y_pred_proba = model.predict_proba(X_val)[:, 1]

    fold_predictions = pd.DataFrame({
        "user_id": val_df["user_id"].values,
        "order_id": val_df["order_id"].values,
        "fold": fold_number,
        "y_true": val_df["correct"].values,
        "y_pred_proba": y_pred_proba,
        "model": "V3.1"
    })

    v31_predictions.append(fold_predictions)

    print(f"Fold {fold_number} predictions generated.")

v31_predictions_df = pd.concat(
    v31_predictions,
    ignore_index=True
)

prediction_path = prediction_dir / "v3_1_predictions.csv"

v31_predictions_df.to_csv(prediction_path, index=False)

print("\nV3.1 predictions saved!")
print("Shape:", v31_predictions_df.shape)
print("Location:", prediction_path)

Fold 1 predictions generated.
Fold 2 predictions generated.
Fold 3 predictions generated.
Fold 4 predictions generated.
Fold 5 predictions generated.

V3.1 predictions saved!
Shape: (219983, 6)
Location: c:\Users\rajur\OneDrive\Desktop\next-move-ai\data\processed\model_predictions\v3_1_predictions.csv
